Neural Physics & Geometric Logic
This notebook explores two advanced paradigms where continuous neural networks learn to emulate discrete or combinatorial logic through geometric phase embeddings. It moves from physical relaxation (Kuramoto oscillators) to explicit computational emulation (Torus Logic).

Part 1: Learning Combinatorial Logic from Kuramoto Dynamics
This section investigates whether a neural network can learn the "physics of optimization." It uses the Kuramoto Model—a system of coupled oscillators used to model synchronization—to solve the Max-Cut Problem.

1. The Physical System
The notebook sets up a system of  N  oscillators coupled by a weighted interaction matrix  W .

The Math: The evolution of phase angles  θi  is governed by: $$\frac{d\theta_i}{dt} = \sum_{j} W_{ij} \sin(\theta_j - \theta_i)$$
The Logic: The energy landscape of this physical system maps to the Max-Cut problem. Oscillators naturally try to anti-synchronize (move to opposite sides of the circle) if the weight  Wij  is negative, effectively "solving" the graph partition problem via physics relaxation.
2. The Neural Learner
A Recurrent Neural Network (GRU) is trained to predict the trajectory of these oscillators.

Input: The current phase state  [cos(θ),sin(θ)] .
Basin Loss: A specialized loss function that doesn't just check if the angle is correct, but checks if the logical partition (the combinatorial solution) matches the ground truth.
Result: The AI learns to simulate the physical relaxation process that leads to a combinatorial solution.
Part 2: The Full Torus Logic Computer (TLC)
This section builds a "Virtual Machine" entirely out of geometry. Instead of using binary (0, 1), it embeds a discrete computer onto a Torus (continuous phase angles) using modular arithmetic over a prime field  Z31 .

1. Geometric Primitives
The notebook defines an "Arithmetic Logic Unit" (ALU) that operates on angles:

Integers as Phases: An integer  x  is mapped to an angle  θ=2πxp .
Multiplication as Rotation: By using discrete logarithms and primitive roots, multiplication  a×b  is transformed into addition in the exponent space. Geometrically, this means multiplication is performed by rotating vectors on the unit circle.
[Image of modular arithmetic clock]

2. The Virtual Machine (VM)
A complete, Turing-like computing architecture is defined with:

Registers & Memory: Stores data as phase angles.
Instruction Set:
STORE: Move data to memory.
MUL: Modular multiplication (via rotation).
ADD: Modular addition.
DEC / JNZ: Decrement and Jump-if-Not-Zero (Control flow).
The Program: A loop that calculates  baseimodp  (modular exponentiation).
3. The AI Emulator (StateRNN)
A GRU-based network is trained to act as the CPU.

Training: The model observes traces of the discrete VM executing code.
Inference: The model is given an initial state and must "hallucinate" the entire execution of the program, updating registers and memory purely through continuous vector operations.
Validation: The output shows a side-by-side comparison of the AI's predicted memory state versus the actual mathematical ground truth. A "PASS" indicates the Neural Network successfully emulated the discrete logic of the computer program.

In [1]:
# @title Part 1: Learning Combinatorial Logic from Kuramoto Dynamics
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. System Configuration ---
np.random.seed(0)
torch.manual_seed(0)

N_nodes = 6
# Generate random symmetric interaction matrix
W = np.random.randn(N_nodes, N_nodes)
W = 0.5 * (W + W.T)

def maxcut_value(theta):
    """Compute the cut value based on the sign of cos(theta)."""
    spins = np.sign(np.cos(theta))
    cut = 0
    for i in range(N_nodes):
        for j in range(i + 1, N_nodes):
            if spins[i] != spins[j]:
                cut += abs(W[i, j])
    return cut

def kuramoto_step(theta, dt=0.05):
    """Update phase angles based on Kuramoto coupling."""
    theta_next = theta.copy()
    N = len(theta)
    for i in range(N):
        d = 0
        for j in range(N):
            if i != j:
                d += np.sin(theta[j] - theta[i])
        theta_next[i] += dt * d
    # Wrap to (-pi, pi]
    theta_next = (theta_next + np.pi) % (2 * np.pi) - np.pi
    return theta_next

def generate_sequence(T=20):
    """Generate a time-series trajectory of phases."""
    theta0 = (np.random.rand(N_nodes) * 2 * np.pi - np.pi)
    seq = [theta0]
    th = theta0
    for _ in range(T):
        th = kuramoto_step(th)
        seq.append(th)
    return np.stack(seq)

def encode(theta):
    """Encode angles as [cos(theta), sin(theta)]."""
    return np.concatenate([np.cos(theta), np.sin(theta)], axis=-1)

# --- 2. Dataset Generation ---
print("Generating Kuramoto trajectories...")
Tseq = 20
ntrain = 400
train_seqs = [generate_sequence(Tseq) for _ in range(ntrain)]

train_X = [encode(s[:-1]) for s in train_seqs]
train_Y = [encode(s[1:])  for s in train_seqs]

# --- 3. Model Definition ---
class RNN(nn.Module):
    def __init__(self, inp=12, hid=64, out=12):
        super().__init__()
        self.rnn = nn.GRU(inp, hid, batch_first=True)
        self.fc = nn.Linear(hid, out)

    def forward(self, x, h=None):
        o, h = self.rnn(x, h)
        return self.fc(o), h

device = "cuda" if torch.cuda.is_available() else "cpu"
model = RNN().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)

def basin_loss(theta_true, theta_pred):
    """Loss based on the difference in MaxCut value of the final state."""
    ct = maxcut_value(theta_true)
    cp = maxcut_value(theta_pred)
    return abs(ct - cp)

# --- 4. Training Loop ---
print(f"Training on {device}...")
epochs = 40
pbar = tqdm(range(epochs))

for epoch in pbar:
    model.train()
    total_loss = 0
    for i in range(ntrain):
        x = torch.tensor(train_X[i], dtype=torch.float32).unsqueeze(0).to(device)
        y = torch.tensor(train_Y[i], dtype=torch.float32).unsqueeze(0).to(device)

        opt.zero_grad()
        yhat, _ = model(x)
        mse = ((yhat - y)**2).mean()

        # Basin-aware: compare final predicted angle logic
        with torch.no_grad():
            last_pred = yhat[0, -1].cpu().numpy()

        # Decode predicted geometry to angle
        c = last_pred[:N_nodes]
        s = last_pred[N_nodes:]
        theta_p = np.arctan2(s, c)

        # True final angle
        last_true = y[0, -1].cpu().numpy()
        ct = last_true[:N_nodes]
        st = last_true[N_nodes:]
        theta_t = np.arctan2(st, ct)

        # Calculate geometric/logic loss
        bl = basin_loss(theta_t, theta_p)
        loss = mse + 0.01 * bl  # Weighted loss
        loss.backward()
        opt.step()
        total_loss += loss.item()

    pbar.set_description(f"Loss: {total_loss/ntrain:.5f}")

# --- 5. Evaluation ---
def rollout_match():
    # Generate long sequence (200 steps) to test attractor stability
    seq = generate_sequence(200)
    theta_true_final = seq[-1]

    # Neural Rollout
    th = seq[0]
    h = None
    for t in range(200):
        x = encode(th)[None, None, :]
        x = torch.tensor(x, dtype=torch.float32).to(device)
        yhat, h = model(x, h)
        yv = yhat[0, 0].detach().cpu().numpy()
        c, s = yv[:N_nodes], yv[N_nodes:]
        th = np.arctan2(s, c)

    ct = maxcut_value(theta_true_final)
    cp = maxcut_value(th)
    return ct, cp

print("\nEvaluating Logic Retention (Attractor Matching)...")
matches = 0
tests = 40
pairs = []
for _ in range(tests):
    ct, cp = rollout_match()
    pairs.append((ct, cp))
    if abs(ct - cp) < 1e-3:
        matches += 1

print(f"Success Rate: {matches}/{tests} ({matches/tests*100}%)")
print("Sample (True Cut, Predicted Cut):", pairs[:5])

Generating Kuramoto trajectories...
Training on cuda...


Loss: 0.00100: 100%|██████████| 40/40 [00:19<00:00,  2.09it/s]



Evaluating Logic Retention (Attractor Matching)...
Success Rate: 40/40 (100.0%)
Sample (True Cut, Predicted Cut): [(0, 0), (0, 0), (0, 0), (0, 0), (0, 0)]


In [2]:
# @title Part 2: The Full Torus Logic Computer (TLC) Learner
import math
import random
from copy import deepcopy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# --- 1. Configuration ---
p = 31              # Prime modulus (Arithmetic on Z_31)
n_harm = 2          # Number of Fourier harmonics (richer geometry)
N_REGS = 5          # Registers
MEM_SIZE = 8        # Memory slots
NUM_RUNS = 512      # Training traces
EPOCHS = 600        # Training epochs
HIDDEN_DIM = 512    # GRU Hidden Dimension
N_LAYERS = 2        # GRU Layers
LR = 1e-3
LAMBDA_CIRCLE = 0.1 # Regularization to keep predictions on unit circle

# --- 2. Geometric Primitives ---
def int_to_phase(x, mod):
    return 2 * math.pi * (x % mod) / mod

def wrap_angle(a):
    return math.atan2(math.sin(a), math.cos(a))

def simulate_branch_logic(theta, p=p):
    """
    Logic gate as phase attraction:
    - Near 0 -> ZERO basin
    - Else -> NONZERO basin
    """
    d = abs(wrap_angle(theta))
    threshold = (2 * math.pi) / p
    branch = "ZERO" if d < threshold else "NONZERO"
    phi_final = 0.0 if branch == "ZERO" else math.pi
    return phi_final, branch

class ExponentALU:
    """
    Implements modular multiplication as addition in exponent space.
    Multiplication becomes rotation.
    """
    def __init__(self, p):
        self.p = p
        self.g = self._find_primitive_root()
        self.log_table, self.exp_table = self._build_tables()

    def _find_primitive_root(self):
        phi = self.p - 1
        factors = set()
        n = phi
        d = 2
        while d * d <= n:
            while n % d == 0:
                factors.add(d); n //= d
            d += 1
        if n > 1: factors.add(n)
        for g in range(2, self.p):
            ok = True
            for q in factors:
                if pow(g, phi // q, self.p) == 1:
                    ok = False; break
            if ok: return g
        raise RuntimeError("No primitive root found")

    def _build_tables(self):
        log_table = [-1] * self.p
        exp_table = [0] * (self.p - 1)
        x = 1
        for k in range(self.p - 1):
            exp_table[k] = x; log_table[x] = k
            x = (x * self.g) % self.p
        return log_table, exp_table

    def mul(self, a, b):
        a %= self.p; b %= self.p
        if a == 0 or b == 0: return 0
        s = (self.log_table[a] + self.log_table[b]) % (self.p - 1)
        return self.exp_table[s]

alu = ExponentALU(p)

# --- 3. TLC Virtual Machine ---
# Program: Calculate base^i mod p
PROG = [
    {"op": "STORE", "addr_reg": 1, "src": 2},   # mem[i] = pow
    {"op": "MUL",   "dst": 2, "a": 2, "b": 0},  # pow *= base
    {"op": "ADD",   "dst": 1, "a": 1, "b": 3},  # i += one
    {"op": "DEC",   "reg": 4},                  # diff -= 1
    {"op": "JNZ",   "reg": 4, "target": 0},     # if diff != 0: loop
]
PROG_LEN = len(PROG)

def init_state(base, N):
    regs = [0]*N_REGS
    regs[0] = base % p; regs[1] = 0; regs[2] = 1; regs[3] = 1; regs[4] = N % p
    return {"regs": regs, "mem": [0]*MEM_SIZE, "pc": 0, "halted": False}

def step_vm(state):
    if state["halted"]: return deepcopy(state)
    regs, mem, pc = state["regs"][:], state["mem"][:], state["pc"]

    if pc < 0 or pc >= PROG_LEN: return {**state, "halted": True}

    inst = PROG[pc]
    op = inst["op"]
    pc_next = pc + 1
    halted = False

    if op == "STORE": mem[regs[inst["addr_reg"]] % MEM_SIZE] = regs[inst["src"]] % p
    elif op == "MUL": regs[inst["dst"]] = alu.mul(regs[inst["a"]], regs[inst["b"]])
    elif op == "ADD": regs[inst["dst"]] = (regs[inst["a"]] + regs[inst["b"]]) % p
    elif op == "DEC": r = inst["reg"]; regs[r] = (regs[r] - 1) % p
    elif op == "JNZ":
        # Logic is determined by phase geometry
        theta = int_to_phase(regs[inst["reg"]], p)
        _, branch = simulate_branch_logic(theta, p)
        if branch == "NONZERO": pc_next = inst["target"]
        else: halted = True # Simple Halt condition for this demo program

    return {"regs": regs, "mem": mem, "pc": pc_next, "halted": halted}

# --- 4. Embedding / Decoding ---
def encode_int_multi_harm(x, mod, n_harm):
    theta = int_to_phase(x, mod)
    out = []
    for k in range(1, n_harm + 1):
        kt = k * theta
        out.extend([math.cos(kt), math.sin(kt)])
    return out

def encode_state(state):
    feats = []
    for r in state["regs"]: feats.extend(encode_int_multi_harm(r, p, n_harm))
    for m in state["mem"]:  feats.extend(encode_int_multi_harm(m, p, n_harm))
    feats.extend(encode_int_multi_harm(state["pc"], PROG_LEN, n_harm))
    return torch.tensor(feats, dtype=torch.float32)

def decode_slot(feats, idx, mod):
    # Decode using first harmonic only
    c, s = feats[idx], feats[idx+1]
    theta = math.atan2(s, c)
    if theta < 0: theta += 2 * math.pi
    y = int(round(theta * mod / (2 * math.pi))) % mod
    return y, idx + 2 * n_harm

def decode_state(vec):
    arr = vec.detach().cpu().numpy().tolist()
    idx = 0
    regs = []; mem = []
    for _ in range(N_REGS): y, idx = decode_slot(arr, idx, p); regs.append(y)
    for _ in range(MEM_SIZE): y, idx = decode_slot(arr, idx, p); mem.append(y)
    pc, idx = decode_slot(arr, idx, PROG_LEN)
    return {"regs": regs, "mem": mem, "pc": pc, "halted": False}

# Determine state dim
tmp = init_state(2, MEM_SIZE)
STATE_DIM = encode_state(tmp).shape[-1]

# --- 5. Dataset Generation ---
print(f"Generating {NUM_RUNS} traces...")
def generate_trace(base, N, max_steps=64):
    state = init_state(base, N)
    states = [state]
    for _ in range(max_steps):
        if state["halted"]: break
        state = step_vm(state)
        states.append(state)
    return states

traces = []
HARD_BASES = [3, 4, 5, 19]
for i in range(NUM_RUNS):
    base = random.choice(HARD_BASES) if random.random() < 0.5 else random.randint(2, p-1)
    tr = generate_trace(base, random.randint(1, MEM_SIZE))
    if len(tr) >= 2:
        enc_tr = torch.stack([encode_state(s) for s in tr])
        traces.append(enc_tr)

# --- 6. Neural Model (StateRNN) ---
class StateRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, h=None):
        # Predict residual ΔS
        out, h = self.gru(x, h)
        delta = self.fc(out)
        return x + delta, h

device = "cuda" if torch.cuda.is_available() else "cpu"
model = StateRNN(STATE_DIM, HIDDEN_DIM, N_LAYERS).to(device)
opt = optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()

def unit_circle_loss(pred):
    # Regularize geometry to stay on circle
    B, T, D = pred.shape
    flat = pred.reshape(B*T, D)
    c, s = flat[:, 0::2], flat[:, 1::2]
    return ((c**2 + s**2 - 1.0)**2).mean()

# --- 7. Training ---
print(f"Training Full TLC Learner on {device}...")
traces_gpu = [tr.to(device) for tr in traces]
pbar = tqdm(range(EPOCHS))

for ep in pbar:
    model.train()
    perm = torch.randperm(len(traces_gpu))
    total_loss = 0
    batch_size = 16

    for i in range(0, len(traces_gpu), batch_size):
        batch = [traces_gpu[j] for j in perm[i:i+batch_size]]
        batch_loss = 0
        for tr in batch:
            x, y = tr[:-1].unsqueeze(0), tr[1:].unsqueeze(0)
            pred, _ = model(x)
            loss = mse(pred, y) + LAMBDA_CIRCLE * unit_circle_loss(pred)
            batch_loss += loss

        if batch:
            batch_loss /= len(batch)
            opt.zero_grad()
            batch_loss.backward()
            opt.step()
            total_loss += batch_loss.item()

    pbar.set_description(f"Loss: {total_loss / (len(traces)/batch_size):.5f}")

# --- 8. Validation & Testing ---
print("\n=== Running AI-as-Computer Verification ===")

def ai_run(base):
    state = init_state(base, MEM_SIZE)
    h = None
    out = []
    for _ in range(40):
        out.append(deepcopy(state))
        if state["halted"]: break

        # Neural Step
        enc = encode_state(state).to(device).unsqueeze(0).unsqueeze(0)
        with torch.no_grad():
            pred_seq, h = model(enc, h)

        # Decode back to discrete to check accuracy
        state = decode_state(pred_seq[0, 0])

        # Check halt condition from decoded PC
        if state["pc"] >= PROG_LEN or state["pc"] < 0:
            state["halted"] = True

    return out

def true_run(base):
    state = init_state(base, MEM_SIZE)
    out = [deepcopy(state)]
    for _ in range(40):
        if state["halted"]: break
        state = step_vm(state)
        out.append(deepcopy(state))
    return out

test_bases = [2, 3, 4, 5, 7, 11, 13, 17, 19, 23, 29]
print(f"{'BASE':<5} | {'MEM MATCH':<10} | {'STEPS (AI/True)':<15} | {'FINAL MEMORY'}")
print("-" * 70)

for base in test_bases:
    ai_trace = ai_run(base)
    true_trace = true_run(base)

    ai_mem = ai_trace[-1]["mem"]
    true_mem = true_trace[-1]["mem"]

    match = (ai_mem == true_mem)
    status = "✅ PASS" if match else "❌ FAIL"

    print(f"{base:<5} | {status:<10} | {len(ai_trace)-1}/{len(true_trace)-1:<13} | {ai_mem}")

Generating 512 traces...
Training Full TLC Learner on cuda...


Loss: 0.00003: 100%|██████████| 600/600 [07:33<00:00,  1.32it/s]


=== Running AI-as-Computer Verification ===
BASE  | MEM MATCH  | STEPS (AI/True) | FINAL MEMORY
----------------------------------------------------------------------
2     | ❌ FAIL     | 39/40            | [1, 2, 4, 8, 16, 1, 3, 2]
3     | ✅ PASS     | 39/40            | [1, 3, 9, 27, 19, 26, 16, 17]
4     | ✅ PASS     | 39/40            | [1, 4, 16, 2, 8, 1, 4, 16]
5     | ✅ PASS     | 39/40            | [1, 5, 25, 1, 5, 25, 1, 5]
7     | ❌ FAIL     | 39/40            | [1, 7, 18, 2, 14, 5, 3, 25]
11    | ✅ PASS     | 39/40            | [1, 11, 28, 29, 9, 6, 4, 13]
13    | ❌ FAIL     | 39/40            | [1, 14, 14, 27, 10, 6, 15, 24]
17    | ✅ PASS     | 39/40            | [1, 17, 10, 15, 7, 26, 8, 12]
19    | ✅ PASS     | 39/40            | [1, 19, 20, 8, 28, 5, 2, 7]
23    | ✅ PASS     | 39/40            | [1, 23, 2, 15, 4, 30, 8, 29]
29    | ❌ FAIL     | 39/40            | [1, 29, 4, 23, 16, 30, 2, 28]
